Tensorflow

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train = x_train.reshape(x_train.shape[0], -1)
X_test = x_test.reshape(x_test.shape[0], -1)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
X_train shape: (60000, 784)
X_test shape: (10000, 784)


In [3]:
def build_model(input_shape, neurons1=128, neurons2=64, dropout_rate=0.2):
    model = Sequential([
        Dense(neurons1, activation='relu', input_shape=(input_shape,)),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(neurons2, activation='relu'),
        BatchNormalization(),
        Dropout(dropout_rate),

        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [4]:
model = build_model(X_train.shape[1])

start_time = time.time()

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    verbose=1
)

end_time = time.time()
total_time = end_time - start_time

y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

report = classification_report(y_test, y_pred)
print(f"Metrics: {report}")

print(f"Время обучения: {total_time/60:.2f} минут")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - accuracy: 0.8440 - loss: 0.5265 - val_accuracy: 0.9407 - val_loss: 0.2043
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9322 - loss: 0.2300 - val_accuracy: 0.9566 - val_loss: 0.1522
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9464 - loss: 0.1758 - val_accuracy: 0.9608 - val_loss: 0.1357
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9558 - loss: 0.1467 - val_accuracy: 0.9634 - val_loss: 0.1276
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9605 - loss: 0.1267 - val_accuracy: 0.9685 - val_loss: 0.1165
Epoch 6/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9655 - loss: 0.1107 - val_accuracy: 0.9680 - val_loss: 0.1158
Epoch 7/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9695 - loss: 0.0983 - val_accuracy: 0.9714 - val_loss: 0.1098
Epoch 8/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9722 - loss: 0.0880 - val_accuracy: 0

Pytorch

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 454kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.27MB/s]


In [6]:
class NeuralNet2(nn.Module):
    def __init__(self, input_size=784, hidden_size1=128, hidden_size2=64, num_classes=10):
        super(NeuralNet2, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.ln1 = nn.LayerNorm(hidden_size1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.ln2 = nn.LayerNorm(hidden_size2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
        self.fc3 = nn.Linear(hidden_size2, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        out = self.fc1(x)
        out = self.ln1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.ln2(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc3(out)
        return out

model = NeuralNet2()

num_epochs = 20
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [7]:
total_step = len(train_loader)
loss_history = []
start_time = time.time()
for epoch in range(num_epochs):
    epoch_loss = 0
    for features, labels in train_loader:

        outputs = model(features)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)

    print('Epoch [{}/{}], Loss: {:.4f}'.format(epoch+1, num_epochs, avg_loss))

end_time = time.time()
total_time = end_time - start_time

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_loader:
        outputs = model(features)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(all_labels, all_preds)
print("Метрики")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print(f"Время обучения: {total_time/60:.2f} минут")

Epoch [1/20], Loss: 0.5192
Epoch [2/20], Loss: 0.2008
Epoch [3/20], Loss: 0.1567
Epoch [4/20], Loss: 0.1328
Epoch [5/20], Loss: 0.1127
Epoch [6/20], Loss: 0.1059
Epoch [7/20], Loss: 0.0956
Epoch [8/20], Loss: 0.0883
Epoch [9/20], Loss: 0.0813
Epoch [10/20], Loss: 0.0761
Epoch [11/20], Loss: 0.0739
Epoch [12/20], Loss: 0.0705
Epoch [13/20], Loss: 0.0682
Epoch [14/20], Loss: 0.0652
Epoch [15/20], Loss: 0.0570
Epoch [16/20], Loss: 0.0557
Epoch [17/20], Loss: 0.0545
Epoch [18/20], Loss: 0.0532
Epoch [19/20], Loss: 0.0518
Epoch [20/20], Loss: 0.0523
Метрики
Accuracy: 0.9808 (98.08%)

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       0.99      0.99      0.99      1135
           2       0.98      0.99      0.98      1032
           3       0.98      0.98      0.98      1010
           4       0.98      0.98      0.98       982
           5       0.99      0.98      0.98       892
           6